In [ ]:
# =============================================================
# 02_clean_merge_panel.ipynb
# Step 2
# ------------------------------------------------------------
# What this does
# ──────────────
# 1. Loads configs from configs/indicators.yml + configs/pipeline.yml
# 2. Reads the coverage matrix from Step 1 to identify/validate
#    the 12-country panel (or accepts MANUAL_12 override)
# 3. Fetches fresh WDI / WGI / UIS data for the 12 countries only
# 4. Merges all three sources (UIS primary, WDI fallback for gaps)
# 5. Applies §3 pipeline preprocessing:
#      - Percentage clipping  [0, 100]
#      - GPI truncation       [0.7, 1.3]
#      - GDP/WGI domain clips
#      - Local linear interpolation for gaps ≤ 2 consecutive years
#      - Binary interpolation mask M_it (1=observed, 0=interpolated)
#      - Long gaps > 2 years left as NaN (native missingness preserved)
# 6. Saves:
#      data/processed/panel_12countries_2015_2024.csv
#      data/processed/panel_12countries_2015_2024.parquet
#      data/processed/interpolation_mask_2015_2024.csv
#      outputs/figures/panel_12/availability_heatmap_*.png
#
# Usage
# ─────
#   python src/02_clean_merge_panel.py
#
#   To override country selection, edit MANUAL_12 below.
#
# Requirements
# ────────────
#   pip install requests pandas numpy matplotlib openpyxl pyyaml
# =============================================================

from __future__ import annotations

import sys
from pathlib import Path

# Notebook-safe path setup
try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    _SRC = _cwd / "src" if (_cwd / "src").exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from utils import (
    find_project_root,
    load_indicators,
    load_pipeline,
    by_source,
    clip_map,
    sort_by_income,
    wb_get_countries,
    build_iso2_map,
    wb_fetch_indicator,
    uis_fetch,
    deduplicate,
    INCOME_ORDER,
)


# ──────────────────────────────────────────────────────────────
# Manual override — set to None to use coverage-based selection
# ──────────────────────────────────────────────────────────────
MANUAL_12: list[str] | None = [
    "ESP", "URY", "CRI",   # High income
    "GEO", "IDN", "THA",   # Upper middle income
    "KGZ", "UZB", "BGD",   # Lower middle income
    "BFA", "BDI", "RWA",   # Low income
]


# ──────────────────────────────────────────────────────────────
# Setup
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
IND_CFG      = load_indicators()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
YEARS      = list(range(YEAR_MIN, YEAR_MAX + 1))
N_YEARS    = len(YEARS)
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"

WB_CFG  = PIPE_CFG["worldbank"]
UIS_CFG = PIPE_CFG["uis"]

ALL_INDICATORS = list(IND_CFG["indicators"].keys())
CLIPS          = clip_map(IND_CFG)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGDIR   = PROJECT_ROOT / "outputs" / "figures" / "panel_12"
COV_DIR  = PROJECT_ROOT / "outputs" / "coverage_analytics"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

print(f"[info] Project root  : {PROJECT_ROOT}")
print(f"[info] Year range    : {YEAR_MIN}–{YEAR_MAX}")
print(f"[info] Indicators    : {len(ALL_INDICATORS)}")
print(f"[info] UIS mode      : {UIS_CFG.get('mode','api').upper()}")
print(f"[info] Manual panel  : {MANUAL_12 is not None}")


# ──────────────────────────────────────────────────────────────
# Step 1 — Resolve 12-country panel
# ──────────────────────────────────────────────────────────────
def resolve_panel(countries_df: pd.DataFrame) -> list[tuple[str, str, str]]:
    """
    Returns list of (iso3c, country_name, income_group) for the 12 panel
    countries, either from MANUAL_12 or from the coverage matrix top-3.

    Returns
    -------
    List of (iso3c, name, income_group) tuples in income-tier order.
    """
    meta = countries_df.set_index("iso3c")[["country", "income_group"]].to_dict("index")

    if MANUAL_12:
        panel = []
        for iso in MANUAL_12:
            iso = iso.upper()
            if iso in meta:
                panel.append((iso, meta[iso]["country"], meta[iso]["income_group"]))
            else:
                print(f"  [warn] {iso} not found in World Bank country list — skipping")
        print(f"\n[step 1] Panel resolved from MANUAL_12 ({len(panel)} countries):")
    else:
        # Load coverage matrix and take top-3 per income tier
        cov_path = COV_DIR / f"top3_by_income_group_{PERIOD_TAG}.csv"
        if not cov_path.exists():
            raise FileNotFoundError(
                f"Coverage matrix not found: {cov_path}\n"
                "Run 01_download_sources.py first."
            )
        top3 = pd.read_csv(cov_path)
        panel = []
        for tier in PIPE_CFG["selection"]["tiers"]:
            for iso in top3.loc[top3["income_group"] == tier, "iso3c"]:
                iso = str(iso).upper()
                if iso in meta:
                    panel.append((iso, meta[iso]["country"], meta[iso]["income_group"]))
        print(f"\n[step 1] Panel resolved from coverage top-3 ({len(panel)} countries):")

    for iso, name, tier in panel:
        print(f"  {iso:<5} {name:<30} {tier}")

    if len(panel) != 12:
        print(f"  [warn] Expected 12 countries, got {len(panel)}")

    return panel


# ──────────────────────────────────────────────────────────────
# Step 2 — Fetch data for 12 countries only
# ──────────────────────────────────────────────────────────────
def fetch_panel_data(
    panel:       list[tuple[str, str, str]],
    countries_df: pd.DataFrame,
    iso2_map:    dict,
    iso_to_name: dict,
) -> pd.DataFrame:
    """
    Pull all 17 indicators for the 12-country panel from WDI / WGI / UIS.
    Returns a raw long-format DataFrame with columns:
        iso3c, country, nice, year, value, _source
    """
    iso_list = [iso for iso, _, _ in panel]

    wdi_specs = by_source(IND_CFG, "wdi")
    wgi_specs = by_source(IND_CFG, "wgi")
    uis_specs = by_source(IND_CFG, "uis")

    frames: list[pd.DataFrame] = []

    print(f"\n[step 2] Fetching data for {len(iso_list)} countries …")

    # ── WDI ─────────────────────────────────────────────────
    print(f"\n  ── WDI ({len(wdi_specs)} indicators) ──")
    for nice, spec in wdi_specs.items():
        df = wb_fetch_indicator(
            iso_list, nice, spec["code"], YEAR_MIN, YEAR_MAX,
            iso2_map, WB_CFG, source=None,
        )
        df["_source"] = "wdi"
        print(f"  [wdi] {nice:<22} rows: {len(df)}")
        frames.append(df)

    # ── WGI ─────────────────────────────────────────────────
    print(f"\n  ── WGI ({len(wgi_specs)} indicators) ──")
    for nice, spec in wgi_specs.items():
        df = wb_fetch_indicator(
            iso_list, nice, spec["code"], YEAR_MIN, YEAR_MAX,
            iso2_map, WB_CFG, source="75",
        )
        df["_source"] = "wgi"
        print(f"  [wgi] {nice:<22} rows: {len(df)}")
        frames.append(df)

    # ── UIS ─────────────────────────────────────────────────
    print(f"\n  ── UIS ({len(uis_specs)} indicators) ──")
    for nice, spec in uis_specs.items():
        bdds_file = spec.get("bdds_file", "both")
        fallback  = spec.get("wb_fallback")

        df = uis_fetch(
            iso_list, nice, spec["code"], YEAR_MIN, YEAR_MAX,
            UIS_CFG, PROJECT_ROOT,
            bdds_file=bdds_file,
            iso_to_name=iso_to_name,
        )
        df["_source"] = "uis"

        if fallback:
            # Fill country-years missing from UIS with WDI
            df_wb = wb_fetch_indicator(
                iso_list, nice, fallback, YEAR_MIN, YEAR_MAX,
                iso2_map, WB_CFG, source=None,
            )
            df_wb["_source"] = "wdi_fallback"
            uis_keys = set(zip(df["iso3c"], df["year"]))
            df_fill  = df_wb[
                ~df_wb.apply(lambda r: (r["iso3c"], r["year"]) in uis_keys, axis=1)
            ].copy()
            df = pd.concat([df, df_fill], ignore_index=True)

        print(f"  [uis] {nice:<22} rows: {len(df)}")
        frames.append(df)

    raw = pd.concat(frames, ignore_index=True)
    raw["avail"] = raw["value"].notna().astype(int)
    raw = deduplicate(raw)
    print(f"\n  Total rows (12 countries, after dedup): {len(raw):,}")
    return raw


# ──────────────────────────────────────────────────────────────
# Step 3 — Build wide panel
# ──────────────────────────────────────────────────────────────
def build_wide_panel(
    raw:   pd.DataFrame,
    panel: list[tuple[str, str, str]],
) -> pd.DataFrame:
    """
    Pivot raw long data to wide country × year format.
    Returns DataFrame with columns: iso3c, country, income_group, year,
    then one column per indicator.
    """
    panel_wide = (
        raw.pivot_table(
            index=["iso3c", "country", "year"],
            columns="nice",
            values="value",
            aggfunc="first",
        )
        .reset_index()
    )

    # Ensure all indicator columns present
    for name in ALL_INDICATORS:
        if name not in panel_wide.columns:
            panel_wide[name] = np.nan

    # Attach income group from panel list
    ig_map = {iso: ig for iso, _, ig in panel}
    panel_wide.insert(
        panel_wide.columns.get_loc("year"),
        "income_group",
        panel_wide["iso3c"].map(ig_map),
    )

    panel_wide = (
        panel_wide
        .assign(_isort=lambda d: d["income_group"].map(INCOME_ORDER).fillna(4))
        .sort_values(["_isort", "iso3c", "year"])
        .drop(columns="_isort")
        .reset_index(drop=True)
    )

    # Reorder columns: identifiers first, then indicators in config order
    id_cols  = ["iso3c", "country", "income_group", "year"]
    ind_cols = [c for c in ALL_INDICATORS if c in panel_wide.columns]
    panel_wide = panel_wide[id_cols + ind_cols]

    print(f"\n[step 3] Wide panel: {len(panel_wide)} rows × {len(panel_wide.columns)} columns")
    print(f"  Countries : {panel_wide['iso3c'].nunique()}")
    print(f"  Years     : {panel_wide['year'].min()}–{panel_wide['year'].max()}")
    print(f"  Indicators: {len(ind_cols)}")
    return panel_wide


# ──────────────────────────────────────────────────────────────
# Step 4 — Apply §3 preprocessing
# ──────────────────────────────────────────────────────────────
def apply_clips(panel_wide: pd.DataFrame) -> pd.DataFrame:
    """
    Apply hard domain constraints per indicators.yml clip_min / clip_max.
    Values outside [clip_min, clip_max] are clipped, not removed.
    """
    panel_wide = panel_wide.copy()
    n_clipped  = 0
    for nice, (lo, hi) in CLIPS.items():
        if nice not in panel_wide.columns:
            continue
        col = panel_wide[nice]
        before = col.notna().sum()
        panel_wide[nice] = col.clip(lower=lo, upper=hi)
        # Count values that were actually changed
        changed = (panel_wide[nice] != col).sum()
        if changed > 0:
            n_clipped += changed
    print(f"\n[step 4a] Clipping: {n_clipped} values clipped to domain bounds")
    return panel_wide


def build_interpolation_mask(panel_wide: pd.DataFrame) -> pd.DataFrame:
    """
    Build binary mask M_it:
      1 = empirically observed (non-null before interpolation)
      0 = interpolated or still missing after interpolation

    Returns a DataFrame with the same shape as panel_wide[id_cols + ind_cols].
    """
    id_cols  = ["iso3c", "country", "income_group", "year"]
    ind_cols = [c for c in ALL_INDICATORS if c in panel_wide.columns]

    mask = panel_wide[id_cols].copy()
    for nice in ind_cols:
        mask[nice] = panel_wide[nice].notna().astype(int)
    return mask


def interpolate_short_gaps(
    panel_wide: pd.DataFrame,
    max_gap:    int = 2,
) -> pd.DataFrame:
    """
    Apply local linear interpolation for gaps ≤ max_gap consecutive
    missing years within each country-indicator series.

    Gaps > max_gap are left as NaN (native missingness preserved per §3).
    Extrapolation beyond the observed range is never applied.
    """
    panel_wide = panel_wide.copy()
    ind_cols   = [c for c in ALL_INDICATORS if c in panel_wide.columns]
    n_filled   = 0

    for nice in ind_cols:
        def _interp_series(s: pd.Series) -> pd.Series:
            """
            Interpolate only within runs of consecutive NaNs whose length
            is <= max_gap.  Runs longer than max_gap are left as NaN.

            Implementation:
              1. Find consecutive NaN runs and their lengths.
              2. Create a temporary copy where NaNs in long runs are
                 replaced with a sentinel float so interpolate() skips them.
              3. Run linear interpolation on the scrubbed series.
              4. Restore sentinels back to NaN.
            """
            if s.notna().sum() < 2:
                return s

            is_nan  = s.isna()
            if not is_nan.any():
                return s

            # Label each consecutive run of identical is_nan values
            runs     = (is_nan != is_nan.shift()).cumsum()
            run_lens = is_nan.groupby(runs).transform("sum")

            # Positions that are NaN AND belong to a run > max_gap
            long_gap = is_nan & (run_lens > max_gap)

            # Sentinel: a value far outside any plausible domain
            SENTINEL = -999_999.0
            s_tmp    = s.copy()
            s_tmp[long_gap] = SENTINEL

            # Interpolate (fills short gaps; sentinel values act as anchors)
            s_tmp = s_tmp.interpolate(
                method="linear",
                limit=max_gap,
                limit_direction="both",
                limit_area="inside",
            )

            # Restore long-gap positions to NaN
            s_tmp[long_gap] = np.nan
            return s_tmp

        original_null = panel_wide[nice].isna().sum()
        panel_wide[nice] = (
            panel_wide.groupby("iso3c")[nice]
            .transform(_interp_series)
        )
        remaining_null = panel_wide[nice].isna().sum()
        filled = original_null - remaining_null
        if filled > 0:
            n_filled += filled

    print(f"[step 4b] Interpolation (≤{max_gap} year gaps): {n_filled} values filled")
    print(f"          Long gaps (>{max_gap} years) preserved as NaN")
    return panel_wide


def summarise_missing(panel_wide: pd.DataFrame) -> None:
    """Print a compact missingness summary for the final panel."""
    ind_cols = [c for c in ALL_INDICATORS if c in panel_wide.columns]
    total    = len(panel_wide)

    print(f"\n[step 4c] Post-processing missingness ({total} country-year rows):")
    print(f"  {'Indicator':<22} {'Missing':>8}  {'%':>6}  {'Countries missing'}") 
    print(f"  {'-'*22} {'-'*8}  {'-'*6}  {'-'*30}")

    for nice in ind_cols:
        n_miss  = panel_wide[nice].isna().sum()
        pct     = n_miss / total * 100
        c_miss  = sorted(panel_wide.loc[panel_wide[nice].isna(), "iso3c"].unique())
        c_str   = ", ".join(c_miss[:6]) + ("…" if len(c_miss) > 6 else "")
        print(f"  {nice:<22} {n_miss:>8}  {pct:>5.1f}%  {c_str}")


# ──────────────────────────────────────────────────────────────
# Step 5 — Availability heatmaps
# ──────────────────────────────────────────────────────────────
def plot_availability_heatmap(
    mask:       pd.DataFrame,
    panel:      list[tuple[str, str, str]],
    indicators: list[str],
) -> None:
    """
    Save a single availability heatmap: countries × indicators,
    cell = fraction of 10 years with observed data.
    """
    country_order = [name for _, name, _ in panel]
    ind_cols      = [c for c in indicators if c in mask.columns]

    # Build summary: mean avail per country-indicator
    summary = (
        mask.groupby("country")[ind_cols]
        .mean()
        .reindex([c for c in country_order if c in mask["country"].values])
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(
        summary.values,
        aspect="auto",
        interpolation="nearest",
        vmin=0, vmax=1,
        cmap="RdYlGn",
    )

    ax.set_title(
        f"Data availability — 12-country panel ({YEAR_MIN}–{YEAR_MAX})\n"
        "(green=complete, yellow=partial, red=missing)",
        fontsize=11,
    )
    ax.set_xlabel("Indicator")
    ax.set_ylabel("Country")

    ax.set_xticks(range(len(ind_cols)))
    ax.set_xticklabels(ind_cols, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(summary.index)))
    ax.set_yticklabels(summary.index, fontsize=9)

    # Annotate cells with percentage
    for i in range(summary.shape[0]):
        for j in range(summary.shape[1]):
            val = summary.iloc[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                        fontsize=6.5,
                        color="black" if 0.3 < val < 0.8 else "white")

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Fraction of years observed")
    fig.tight_layout()

    out = FIGDIR / f"availability_heatmap_panel_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n[step 5] Saved: {out.name}")


def plot_outcome_trends(
    panel_wide: pd.DataFrame,
    panel:      list[tuple[str, str, str]],
) -> None:
    """
    Save a 2×2 panel trend plot for the primary outcome variable (sec_completion).

    Layout: one subplot per income tier so high-coverage countries clustered
    at 95–100% no longer compete visually with lower-income trajectories.
    Colours: four hue families (blues/greens/oranges/purples), one per tier,
    three shades within each tier.  White stroke outlines separate overlapping
    lines.  BDI post-2020 data gap annotated explicitly.
    """
    import matplotlib.patheffects as pe

    if "sec_completion" not in panel_wide.columns:
        return

    # ── Per-country visual style ─────────────────────────────────────────────
    STYLE: dict[str, dict] = {
        # High income — blues
        "ESP": {"color": "#1a6fad", "marker": "o", "lw": 2.2, "ms": 7, "ls": "-"},
        "URY": {"color": "#5ba4d4", "marker": "s", "lw": 2.2, "ms": 7, "ls": "-"},
        "CRI": {"color": "#a8d1f0", "marker": "^", "lw": 2.2, "ms": 7, "ls": "-"},
        # Upper middle — greens
        "GEO": {"color": "#1a7a3a", "marker": "o", "lw": 2.2, "ms": 7, "ls": "-"},
        "IDN": {"color": "#4dac26", "marker": "s", "lw": 2.2, "ms": 7, "ls": "-"},
        "THA": {"color": "#a6d96a", "marker": "^", "lw": 2.2, "ms": 7, "ls": "-"},
        # Lower middle — oranges/reds
        "KGZ": {"color": "#d73027", "marker": "o", "lw": 2.2, "ms": 7, "ls": "-"},
        "UZB": {"color": "#f46d43", "marker": "s", "lw": 2.2, "ms": 7, "ls": "-"},
        "BGD": {"color": "#fdae61", "marker": "^", "lw": 2.2, "ms": 7, "ls": "-"},
        # Low income — purples
        "BFA": {"color": "#7b2d8b", "marker": "o", "lw": 2.2, "ms": 7, "ls": "-"},
        "BDI": {"color": "#c994c7", "marker": "x", "lw": 1.8, "ms": 8, "ls": "--"},
        "RWA": {"color": "#4d1f6e", "marker": "D", "lw": 2.2, "ms": 6, "ls": "-"},
    }

    TIER_ORDER = [
        ("High income",         "H"),
        ("Upper middle income", "UM"),
        ("Lower middle income", "LM"),
        ("Low income",          "L"),
    ]
    TIER_KEY = {
        "High income":          "H",
        "Upper middle income":  "UM",
        "Lower middle income":  "LM",
        "Low income":           "L",
    }
    tier_to_isos: dict[str, list[tuple[str, str]]] = {
        "H": [], "UM": [], "LM": [], "L": []
    }
    for iso, name, tier in panel:
        tk = TIER_KEY.get(tier, "L")
        tier_to_isos[tk].append((iso, name))

    years = sorted(panel_wide["year"].unique())

    fig, axes = plt.subplots(
        2, 2,
        figsize=(16, 11),
        sharex=True,
        gridspec_kw={"hspace": 0.38, "wspace": 0.22},
    )
    ax_map = {
        "H":  axes[0, 0],
        "UM": axes[0, 1],
        "LM": axes[1, 0],
        "L":  axes[1, 1],
    }

    for tier_label, tk in TIER_ORDER:
        ax = ax_map[tk]
        bdi_gap_point: tuple | None = None

        for iso, name in tier_to_isos.get(tk, []):
            d     = panel_wide[panel_wide["iso3c"] == iso].sort_values("year")
            style = STYLE.get(iso, {"color": "grey", "marker": "o",
                                    "lw": 1.8, "ms": 6, "ls": "-"})

            # Split series at NaN boundaries → no line drawn across gaps
            seg_x: list[float] = []
            seg_y: list[float] = []

            for _, row in d.iterrows():
                v = row["sec_completion"]
                y = row["year"]
                if pd.notna(v):
                    seg_x.append(y)
                    seg_y.append(v)
                    # Track BDI last observed point before gap
                    if iso == "BDI":
                        bdi_gap_point = (y, v)
                else:
                    if seg_x:
                        ax.plot(
                            seg_x, seg_y,
                            color=style["color"],
                            marker=style["marker"],
                            linewidth=style["lw"],
                            markersize=style["ms"],
                            linestyle=style["ls"],
                            label=f"{iso} ({name})",
                            zorder=3,
                            path_effects=[
                                pe.withStroke(linewidth=3.8, foreground="white")
                            ],
                        )
                        seg_x, seg_y = [], []

            if seg_x:
                ax.plot(
                    seg_x, seg_y,
                    color=style["color"],
                    marker=style["marker"],
                    linewidth=style["lw"],
                    markersize=style["ms"],
                    linestyle=style["ls"],
                    label=f"{iso} ({name})",
                    zorder=3,
                    path_effects=[
                        pe.withStroke(linewidth=3.8, foreground="white")
                    ],
                )

        # BDI data-gap annotation
        if tk == "L" and bdi_gap_point is not None:
            gx, gy = bdi_gap_point
            ax.annotate(
                "BDI: data gap\npost-2020",
                xy=(gx, gy),
                xytext=(gx + 0.8, gy + 22),
                arrowprops=dict(arrowstyle="->", color="#c994c7", lw=1.2),
                fontsize=8,
                color="#7b2d8b",
                bbox=dict(boxstyle="round,pad=0.3", fc="white",
                          ec="#c994c7", alpha=0.9),
                zorder=5,
            )

        ax.set_title(tier_label, fontsize=12, fontweight="bold", pad=8)
        ax.set_xlim(min(years) - 0.5, max(years) + 0.5)
        ax.set_ylim(-2, 115)
        ax.set_xticks(years)
        ax.set_xticklabels(
            [str(int(y)) for y in years],
            rotation=45, ha="right", fontsize=9,
        )
        ax.set_yticks([0, 20, 40, 60, 80, 100])
        ax.yaxis.set_tick_params(labelsize=9)
        ax.axhline(100, color="#cccccc", lw=0.8, ls=":")
        ax.axhline(80,  color="#eeeeee", lw=0.6, ls=":")
        ax.grid(axis="y", alpha=0.25, lw=0.6)
        ax.grid(axis="x", alpha=0.15, lw=0.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(
            loc="upper left",
            bbox_to_anchor=(1.01, 1.0),
            fontsize=9,
            frameon=True,
            framealpha=0.95,
            edgecolor="#cccccc",
            borderpad=0.6,
            labelspacing=0.5,
        )

    # Shared axis labels
    for ax in axes[1]:
        ax.set_xlabel("Year", fontsize=10)
    for ax in axes[:, 0]:
        ax.set_ylabel("Completion rate (%)", fontsize=10)

    fig.suptitle(
        f"Lower-secondary completion rate — 12-country panel "
        f"({YEAR_MIN}–{YEAR_MAX})",
        fontsize=14, fontweight="bold", y=1.01,
    )

    out = FIGDIR / f"trend_sec_completion_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"[step 5] Saved: {out.name}")


# ──────────────────────────────────────────────────────────────
# Step 5b — Combined single-axis trend plot (all 12 countries)
# ──────────────────────────────────────────────────────────────
def plot_outcome_trends_combined(
    panel_wide: pd.DataFrame,
    panel:      list[tuple[str, str, str]],
) -> None:
    """
    Save a single-axis trend plot with all 12 countries overlaid.
    Useful for direct cross-country comparison.  Countries are
    distinguished by colour (tier family) + marker shape.
    """
    import matplotlib.patheffects as pe

    if "sec_completion" not in panel_wide.columns:
        return

    STYLE_COMBINED: dict[str, dict] = {
        "ESP": {"color": "#1a6fad", "marker": "o",  "lw": 2.0, "ms": 6, "ls": "-"},
        "URY": {"color": "#5ba4d4", "marker": "s",  "lw": 2.0, "ms": 6, "ls": "-"},
        "CRI": {"color": "#a8d1f0", "marker": "^",  "lw": 2.0, "ms": 6, "ls": "-"},
        "GEO": {"color": "#1a7a3a", "marker": "o",  "lw": 2.0, "ms": 6, "ls": "-"},
        "IDN": {"color": "#4dac26", "marker": "s",  "lw": 2.0, "ms": 6, "ls": "-"},
        "THA": {"color": "#a6d96a", "marker": "^",  "lw": 2.0, "ms": 6, "ls": "-"},
        "KGZ": {"color": "#d73027", "marker": "o",  "lw": 2.0, "ms": 6, "ls": "-"},
        "UZB": {"color": "#f46d43", "marker": "s",  "lw": 2.0, "ms": 6, "ls": "-"},
        "BGD": {"color": "#fdae61", "marker": "^",  "lw": 2.0, "ms": 6, "ls": "-"},
        "BFA": {"color": "#7b2d8b", "marker": "o",  "lw": 2.0, "ms": 6, "ls": "-"},
        "BDI": {"color": "#c994c7", "marker": "x",  "lw": 1.6, "ms": 7, "ls": "--"},
        "RWA": {"color": "#4d1f6e", "marker": "D",  "lw": 2.0, "ms": 5, "ls": "-"},
    }

    fig, ax = plt.subplots(figsize=(14, 7))

    for iso, name, tier in panel:
        d     = panel_wide[panel_wide["iso3c"] == iso].sort_values("year")
        style = STYLE_COMBINED.get(iso, {"color": "grey", "marker": "o",
                                         "lw": 1.8, "ms": 5, "ls": "-"})
        seg_x: list[float] = []
        seg_y: list[float] = []

        for _, row in d.iterrows():
            v = row["sec_completion"]
            y = row["year"]
            if pd.notna(v):
                seg_x.append(y)
                seg_y.append(v)
            else:
                if seg_x:
                    ax.plot(seg_x, seg_y,
                            color=style["color"], marker=style["marker"],
                            linewidth=style["lw"], markersize=style["ms"],
                            linestyle=style["ls"],
                            label=f"{iso} ({name})",
                            zorder=3,
                            path_effects=[
                                pe.withStroke(linewidth=3.2, foreground="white")
                            ])
                    seg_x, seg_y = [], []
        if seg_x:
            ax.plot(seg_x, seg_y,
                    color=style["color"], marker=style["marker"],
                    linewidth=style["lw"], markersize=style["ms"],
                    linestyle=style["ls"],
                    label=f"{iso} ({name})",
                    zorder=3,
                    path_effects=[
                        pe.withStroke(linewidth=3.2, foreground="white")
                    ])

    # Tier colour legend patches
    from matplotlib.patches import Patch
    tier_patches = [
        Patch(color="#1a6fad", label="High income"),
        Patch(color="#1a7a3a", label="Upper middle income"),
        Patch(color="#d73027", label="Lower middle income"),
        Patch(color="#7b2d8b", label="Low income"),
    ]

    years = sorted(panel_wide["year"].unique())
    ax.set_xlim(min(years) - 0.5, max(years) + 0.5)
    ax.set_ylim(-2, 115)
    ax.set_xticks(years)
    ax.set_xticklabels([str(int(y)) for y in years], fontsize=10)
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    ax.axhline(100, color="#cccccc", lw=0.8, ls=":")
    ax.axhline(80,  color="#eeeeee", lw=0.6, ls=":")
    ax.grid(axis="y", alpha=0.25, lw=0.6)
    ax.grid(axis="x", alpha=0.15, lw=0.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("Completion rate (%)", fontsize=11)
    ax.set_title(
        f"Lower-secondary completion rate — 12-country panel "
        f"({YEAR_MIN}–{YEAR_MAX})",
        fontsize=13, fontweight="bold",
    )

    # Country legend — two columns, outside right
    country_leg = ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        fontsize=9,
        frameon=True,
        framealpha=0.95,
        edgecolor="#cccccc",
        title="Country",
        title_fontsize=9,
        ncol=1,
    )
    ax.add_artist(country_leg)

    # Tier legend — bottom right of axes
    ax.legend(
        handles=tier_patches,
        loc="lower left",
        bbox_to_anchor=(1.01, 0.0),
        fontsize=9,
        frameon=True,
        framealpha=0.95,
        edgecolor="#cccccc",
        title="Income tier",
        title_fontsize=9,
    )

    fig.tight_layout()
    out = FIGDIR / f"trend_sec_completion_combined_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"[step 5] Saved: {out.name}")


# ──────────────────────────────────────────────────────────────
# Step 6 — Save outputs
# ──────────────────────────────────────────────────────────────
def save_outputs(
    panel_wide: pd.DataFrame,
    mask:       pd.DataFrame,
) -> None:
    print("\n[step 6] Saving panel outputs …")

    # Wide panel CSV
    p = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.csv"
    panel_wide.to_csv(p, index=False)
    print(f"  [OK] {p.name}  ({len(panel_wide)} rows × {len(panel_wide.columns)} cols)")

    # Wide panel Parquet
    try:
        p2 = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.parquet"
        panel_wide.to_parquet(p2, index=False)
        print(f"  [OK] {p2.name}")
    except Exception as e:
        print(f"  [skip] Parquet export: {e}  (pip install pyarrow to enable)")

    # Interpolation mask
    p3 = DATA_DIR / f"interpolation_mask_{PERIOD_TAG}.csv"
    mask.to_csv(p3, index=False)
    print(f"  [OK] {p3.name}")

    # Quick panel stats
    ind_cols = [c for c in ALL_INDICATORS if c in panel_wide.columns]
    total    = len(panel_wide)
    obs      = panel_wide[ind_cols].notna().sum().sum()
    cells    = total * len(ind_cols)
    print(f"\n  Panel completeness: {obs:,}/{cells:,} cells "
          f"({obs/cells:.1%}) non-null after interpolation")
    print(f"  Expected balanced panel: {len(MANUAL_12 or []) * N_YEARS} rows "
          f"(12 countries × {N_YEARS} years)")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    # Country metadata
    print("\n[setup] Fetching World Bank country metadata …")
    countries_df = wb_get_countries(WB_CFG)
    iso2_map     = build_iso2_map(countries_df, WB_CFG.get("iso2_supplement", {}))
    iso_to_name  = dict(zip(countries_df["iso3c"], countries_df["country"]))

    # Resolve panel
    panel = resolve_panel(countries_df)
    if not panel:
        raise SystemExit("[fatal] No countries resolved — check MANUAL_12 or coverage matrix.")

    # Fetch data
    raw = fetch_panel_data(panel, countries_df, iso2_map, iso_to_name)

    # Build wide panel
    panel_wide = build_wide_panel(raw, panel)

    # Preprocessing
    panel_wide = apply_clips(panel_wide)
    mask       = build_interpolation_mask(panel_wide)   # mask BEFORE interpolation
    panel_wide = interpolate_short_gaps(panel_wide, max_gap=2)
    summarise_missing(panel_wide)

    # Figures
    ind_cols = [c for c in ALL_INDICATORS if c in panel_wide.columns]
    plot_availability_heatmap(mask, panel, ind_cols)
    plot_outcome_trends(panel_wide, panel)           # 2×2 panel by income tier
    plot_outcome_trends_combined(panel_wide, panel)  # single-axis all countries

    # Save
    save_outputs(panel_wide, mask)

    print(f"\n[done] Panel data : {DATA_DIR.resolve()}")
    print(f"[done] Figures    : {FIGDIR.resolve()}")
    print("\n[next] Run: python src/03_create_masks_and_labels.py")


if __name__ == "__main__":
    main()